# LiTo → human fine-tune (Colab free tier / T4)
Fine-tunes Apple LiTo's 623M DiT on human scans with **LoRA** (free-tier friendly: ~25M trainable params, fp16 AMP, batch 1 + grad-accum, Drive checkpointing for preempted sessions).

**Before you start (one-time):**
1. Request access to [THuman2.1](https://github.com/ytrock/THuman2.0-Dataset) (and/or [2K2K](https://github.com/SangHunHan92/2K2K)) — research license, manual approval.
2. Put textured scans (`.glb`/`.obj`+texture) in Drive: `MyDrive/lito_ft/scans/`. Even 100–300 scans is a meaningful domain fine-tune; start with ~30 to validate the pipeline end-to-end.
3. Runtime → Change runtime type → **T4 GPU**.

Sessions die on free tier — every stage caches to Drive and resumes. Just "Run all" again.

In [ ]:
# ── 1. Environment ──────────────────────────────────────────────────────────
from google.colab import drive; drive.mount('/content/drive')
import os, sys, glob, math, json
WORK = '/content/drive/MyDrive/lito_ft'
for d in ['scans','renders','latents','conds','ckpts','out']: os.makedirs(f'{WORK}/{d}', exist_ok=True)
if not os.path.exists('/content/ml-lito'):
    !git clone -q https://github.com/apple/ml-lito /content/ml-lito
%pip -q install safetensors peft einops timm trimesh pyrender PyOpenGL xformers
sys.path.insert(0, '/content/ml-lito/src')
CKPT = f'{WORK}/lito_dit_rgba.ckpt'   # 7.4 GB — cached on Drive after first download
if not os.path.exists(CKPT):
    !wget -q --show-progress -O {CKPT} https://ml-site.cdn-apple.com/models/lito/lito_dit_rgba.ckpt
import torch; print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── 2. Load LiTo (frozen) ───────────────────────────────────────────────────
# Loads the DiT trainer + tokenizer exactly like the repo's own demo does.
# If Apple moves an API, mirror whatever demos/lito/fastapi_lito_demo.py does here.
import torch
ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
from lito.script_utils import config_utils
cfg = ckpt['hyper_parameters'] if 'hyper_parameters' in ckpt else ckpt['config']
trainer = config_utils.instantiate_from_config(cfg) if isinstance(cfg, dict) and 'target' in cfg else None
if trainer is None:
    # fallback: the repo provides a loader used by its demo
    from demos.lito.fastapi_lito_demo import load_models  # noqa — adjust to repo if renamed
    trainer = load_models(CKPT)
missing = trainer.load_state_dict(ckpt['state_dict'] if 'state_dict' in ckpt else ckpt, strict=False)
print('load:', missing)
trainer = trainer.cuda().eval()
dit  = trainer.velocity_estimator_ema.module      # fine-tune FROM the EMA weights
st   = trainer.pretrained_tokenizer if hasattr(trainer, 'pretrained_tokenizer') else trainer
LAT_MEAN, LAT_STD, T_EPS = float(trainer.latent_mean), float(trainer.latent_std), float(trainer.t_eps)
print('latent stats:', LAT_MEAN, LAT_STD, T_EPS)

In [ ]:
# ── 3. Render multiview RGBD per scan (cached) ──────────────────────────────
os.environ['PYOPENGL_PLATFORM'] = 'egl'
import numpy as np, trimesh, pyrender
V, RES = 48, 518                       # paper uses 150 views; 48 is enough to fine-tune
def fib_sphere(n, r=2.0):
    pts=[]; ga=math.pi*(3-math.sqrt(5))
    for i in range(n):
        y=1-2*i/(n-1); rad=math.sqrt(max(0,1-y*y)); th=ga*i
        pts.append((r*rad*math.cos(th), r*y, r*rad*math.sin(th)))
    return pts
def look_at(eye):
    f=-np.array(eye); f/=np.linalg.norm(f); up=np.array([0,1,0.])
    if abs(f@up)>0.99: up=np.array([0,0,1.])
    s=np.cross(f,up); s/=np.linalg.norm(s); u=np.cross(s,f)
    m=np.eye(4); m[:3,0]=s; m[:3,1]=u; m[:3,2]=-f; m[:3,3]=eye; return m
for path in sorted(glob.glob(f'{WORK}/scans/*')):
    name=os.path.splitext(os.path.basename(path))[0]
    outp=f'{WORK}/renders/{name}.npz'
    if os.path.exists(outp): continue
    m=trimesh.load(path, force='mesh')
    m.apply_translation(-(m.bounds[0]+m.bounds[1])/2)
    m.apply_scale(1.8/max(m.extents))               # fit [-0.9,0.9] like LiTo objects
    scene=pyrender.Scene(ambient_light=[.55,.55,.55])
    scene.add(pyrender.Mesh.from_trimesh(m))
    cam=pyrender.PerspectiveCamera(yfov=0.69)
    cn=scene.add(cam, pose=np.eye(4))
    r=pyrender.OffscreenRenderer(RES,RES)
    rgbs,depths,poses=[],[],[]
    for eye in fib_sphere(V):
        pose=look_at(np.array(eye)); scene.set_pose(cn,pose)
        rgb,dep=r.render(scene)
        rgbs.append(rgb); depths.append(dep); poses.append(pose)
    r.delete()
    np.savez_compressed(outp, rgb=np.stack(rgbs).astype(np.uint8),
                        depth=np.stack(depths).astype(np.float16),
                        pose=np.stack(poses).astype(np.float32), yfov=0.69)
    print('rendered', name)
print('scans rendered:', len(glob.glob(f'{WORK}/renders/*.npz')))

In [ ]:
# ── 4. Tokenize scans + cache conditioning tokens (frozen encoders) ─────────
# Latents: surface-light-field samples (points+dirs+rgb from RGBD) → 8192×32.
# Follow the repo's notebooks/demo_tokenizer.ipynb for the exact encode call —
# the wrapper below matches the trainer API at the time of writing.
import numpy as np, torch
@torch.no_grad()
def encode_scan(npz):
    d=np.load(npz)
    rgb=torch.from_numpy(d['rgb']).float().cuda()/255.    # (V,H,W,3)
    dep=torch.from_numpy(d['depth'].astype(np.float32)).cuda()
    pose=torch.from_numpy(d['pose']).cuda()
    lat = trainer.tokenize_rgbd(rgb=rgb[None], depth=dep[None], H_c2w=pose[None], yfov=float(d['yfov']))
    return lat.squeeze(0).cpu()                            # (8192, 32) unnormalized
@torch.no_grad()
def cond_tokens(npz, view):
    d=np.load(npz)
    rgb=torch.from_numpy(d['rgb'][view]).float().cuda()/255.
    a=(torch.from_numpy((d['depth'][view]>0)).float().cuda())[...,None]
    return trainer.get_image_conditioning(straight_rgb=rgb[None,None], alpha=a[None,None]).squeeze(0).cpu()
for npz in sorted(glob.glob(f'{WORK}/renders/*.npz')):
    name=os.path.splitext(os.path.basename(npz))[0]
    lp, cp = f'{WORK}/latents/{name}.pt', f'{WORK}/conds/{name}.pt'
    if not os.path.exists(lp): torch.save(encode_scan(npz), lp); print('latent', name)
    if not os.path.exists(cp):
        torch.save(torch.stack([cond_tokens(npz,v) for v in range(0,48,6)]), cp)  # 8 cond views/scan
print('latents:', len(glob.glob(f'{WORK}/latents/*.pt')))

In [ ]:
# ── 5. LoRA fine-tune (resumable) ───────────────────────────────────────────
from peft import LoraConfig, get_peft_model
import random, glob, torch, torch.nn.functional as F
STEPS, ACCUM, LR, RANK = 12000, 8, 1e-4, 16
targets=[n for n,_ in dit.named_modules() if n.split('.')[-1] in
         ('linear_qkv','linear_out','linear_q','linear_kv','w1','w2','w3')]
model=get_peft_model(dit, LoraConfig(r=RANK, lora_alpha=RANK, target_modules=targets, bias='none')).cuda()
model.print_trainable_parameters()
opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
scaler=torch.amp.GradScaler(); step=0
resume=sorted(glob.glob(f'{WORK}/ckpts/lora_*.pt'))
if resume:
    s=torch.load(resume[-1]); model.load_state_dict(s['lora'], strict=False)
    opt.load_state_dict(s['opt']); step=s['step']; print('resumed @', step)
names=[os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{WORK}/latents/*.pt')]
y_emb = dit.cond_embedder.y_embedding                     # uncond row for 10% cond dropout (CFG)
model.train()
while step < STEPS:
    opt.zero_grad(set_to_none=True)
    for _ in range(ACCUM):
        n=random.choice(names)
        x1=((torch.load(f'{WORK}/latents/{n}.pt').cuda()-LAT_MEAN)/LAT_STD)[None]
        cs=torch.load(f'{WORK}/conds/{n}.pt'); cond=cs[random.randrange(len(cs))][None].cuda()
        if random.random()<0.1: cond=y_emb[None,None].expand_as(cond).detach()
        x0=torch.randn_like(x1); t=torch.rand(1,device='cuda')*(1-T_EPS)+T_EPS
        xt=(1-t)*x0+t*x1                                    # linear flow path, t=0 noise → 1 data
        with torch.autocast('cuda', dtype=torch.float16):
            v=model(tokens=xt, t=t, cond=cond)
            loss=F.mse_loss(v.float(), (x1-x0))/ACCUM
        scaler.scale(loss).backward()
    scaler.step(opt); scaler.update(); step+=1
    if step%50==0: print(step, f'{loss.item()*ACCUM:.4f}')
    if step%500==0 or step==STEPS:
        torch.save({'lora':{k:v for k,v in model.state_dict().items() if 'lora' in k},
                    'opt':opt.state_dict(),'step':step}, f'{WORK}/ckpts/lora_{step:06d}.pt')
        print('checkpointed', step)

In [ ]:
# ── 6. Merge LoRA + export safetensors for the Swift app ────────────────────
from safetensors.torch import save_file
merged = model.merge_and_unload()                        # bakes LoRA into the DiT
sd = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
new = {k: v.contiguous() for k, v in sd.items()}
for k, v in merged.state_dict().items():
    for prefix in ('velocity_estimator_ema.module.', 'velocity_estimator.'):
        kk = prefix + k
        if kk in new: new[kk] = v.contiguous().float()
out=f'{WORK}/out/lito_humans.safetensors'
save_file({k: v.float() for k, v in new.items()}, out)
print('wrote', out, round(os.path.getsize(out)/1e9,2), 'GB')
print('Install: back up weights/lito.safetensors, copy this file in its place, relaunch the app.')

### Notes
- **Validate cheap first**: run with ~30 scans / `STEPS=1000` end-to-end before committing GPU-days.
- Free-tier T4 ≈ 1.5–2.5 it/s at batch 1×8 accum → 12k steps over ~2–3 sessions (resume is automatic).
- Cells 2/4 wrap Apple's APIs (`tokenize_rgbd`, `get_image_conditioning`); if a name drifted, mirror `notebooks/demo_tokenizer.ipynb` / `demos/lito/fastapi_lito_demo.py` from the repo — the rest of the pipeline is self-contained.
- Don't over-train: humans-only data will degrade generic objects (fine if humans are all you want). Add ~10–20% Objaverse renders to the mix otherwise.
- Licenses: THuman/2K2K are research-only — keep the fine-tuned weights private.